<a href="https://colab.research.google.com/github/UmymaM/ml-dl-cv-fundamentals/blob/main/face-verification/face_verification_w_Siamese_Network.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms, models
from torch.utils.data import DataLoader, random_split
import matplotlib.pyplot as plt
import numpy as np

In [3]:
device=torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using Device: {device}")

Using Device: cuda


In [4]:
!kaggle datasets download -d atulanandjha/lfwpeople

Dataset URL: https://www.kaggle.com/datasets/atulanandjha/lfwpeople
License(s): GNU Lesser General Public License 3.0
lfwpeople.zip: Skipping, found more recently modified local copy (use --force to force download)


In [5]:
import zipfile
zipfile=zipfile.ZipFile('lfwpeople.zip') #unzipping the lfwpeople file
zipfile.extractall()
zipfile.close()

In [6]:
import tarfile

# Path to the tgz file
tgz_file_path = '/content/lfw-funneled.tgz'

# Directory to extract to
extract_dir = '/content/'

# Open the tgz file in read mode ('r:gz')
with tarfile.open(tgz_file_path, 'r:gz') as tar:
    # Extract all contents to the specified directory
    tar.extractall(path=extract_dir)

/tmp/ipykernel_16670/4013497105.py:12: DeprecationWarning: Python 3.14 will, by default, filter extracted tar archives and reject files or modify their metadata. Use the filter argument to control this behavior.
  tar.extractall(path=extract_dir)


In [7]:
train_transform=transforms.Compose([
    transforms.Resize((224,224)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomRotation(degrees=10),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225])
])
val_transform=transforms.Compose([
    transforms.Resize((224,224)),
    transforms.ToTensor(), #no augmentation for test ds
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225])
])

In [8]:
import os

In [9]:
labeled_faces_dict={}
# filtering files
for file in os.listdir('/content/lfw_funneled'):
    file_path = os.path.join('/content/lfw_funneled', file)
    images=[]
    if os.path.isdir(file_path):
        # print(f"Length of dir {file}: {len(os.listdir(file_path))}")
        # continue
        for image in os.listdir(file_path):
          image=os.path.join(file_path,image)
          images.append(image)
        labeled_faces_dict[file]=images
    else:
        print(f"Skipping file: {file}")

Skipping file: pairs_01.txt
Skipping file: pairs_02.txt
Skipping file: pairs.txt
Skipping file: pairs_04.txt
Skipping file: pairs_07.txt
Skipping file: pairs_06.txt
Skipping file: pairs_05.txt
Skipping file: pairs_10.txt
Skipping file: pairs_03.txt
Skipping file: pairs_09.txt
Skipping file: pairs_08.txt


In [10]:
len(labeled_faces_dict.keys())

5749

In [11]:
labeled_faces_dict.get("Charles_Schumer")

['/content/lfw_funneled/Charles_Schumer/Charles_Schumer_0002.jpg',
 '/content/lfw_funneled/Charles_Schumer/Charles_Schumer_0001.jpg']

In [12]:
for idx,file in enumerate(labeled_faces_dict.keys()):
  print(f"{idx}: {file} {len(labeled_faces_dict[file])}")
  if idx==20:
    break

0: Joel_Gallen 1
1: Vidar_Helgesen 2
2: Linda_Amicangioli 1
3: Ronald_Post 1
4: Lucas_Wysocki 1
5: Marquis_Estill 1
6: Jen_Bice 1
7: Kyra_Sedgwick 1
8: Chris_Moore 1
9: Kyle_McLaren 1
10: Mahmoud_Al_Zhar 1
11: Frank_Shea 1
12: Paul_Patton 2
13: Andrew_Caldecott 1
14: Shingo_Suetsugu 1
15: Paul_William_Hurley 2
16: Toby_Keith 1
17: Bill_Mauldin 1
18: Tim_Duncan 4
19: Jean-Marc_de_La_Sabliere 2
20: Jose_Woldenberg 1


In [13]:
from itertools import combinations

In [14]:
positive_pairs=[]
negative_pairs=[]
# making +ve pairs
for person,images in labeled_faces_dict.items():
  if len(images)>1:
      for image1,image2 in combinations(images,2):
        pair=(image1,image2,1)
        positive_pairs.append(pair)

In [15]:
for i in range(10):
  print(f"Positive Pairs: {positive_pairs[i]}")

Positive Pairs: ('/content/lfw_funneled/Vidar_Helgesen/Vidar_Helgesen_0001.jpg', '/content/lfw_funneled/Vidar_Helgesen/Vidar_Helgesen_0002.jpg', 1)
Positive Pairs: ('/content/lfw_funneled/Paul_Patton/Paul_Patton_0001.jpg', '/content/lfw_funneled/Paul_Patton/Paul_Patton_0002.jpg', 1)
Positive Pairs: ('/content/lfw_funneled/Paul_William_Hurley/Paul_William_Hurley_0002.jpg', '/content/lfw_funneled/Paul_William_Hurley/Paul_William_Hurley_0001.jpg', 1)
Positive Pairs: ('/content/lfw_funneled/Tim_Duncan/Tim_Duncan_0002.jpg', '/content/lfw_funneled/Tim_Duncan/Tim_Duncan_0001.jpg', 1)
Positive Pairs: ('/content/lfw_funneled/Tim_Duncan/Tim_Duncan_0002.jpg', '/content/lfw_funneled/Tim_Duncan/Tim_Duncan_0003.jpg', 1)
Positive Pairs: ('/content/lfw_funneled/Tim_Duncan/Tim_Duncan_0002.jpg', '/content/lfw_funneled/Tim_Duncan/Tim_Duncan_0004.jpg', 1)
Positive Pairs: ('/content/lfw_funneled/Tim_Duncan/Tim_Duncan_0001.jpg', '/content/lfw_funneled/Tim_Duncan/Tim_Duncan_0003.jpg', 1)
Positive Pairs: ('/c

In [16]:
import random

In [17]:
type(labeled_faces_dict.keys())

dict_keys

In [18]:
# making negative pairs
# for a balanced ds, we'll iterate the loop the same number of times as the+ve pairs
for i in range(len(positive_pairs)):
  # generating 2 random names from the dict
  n1,n2=random.sample(list(labeled_faces_dict.keys()),2)
  image1=random.choice(labeled_faces_dict[n1])
  image2=random.choice(labeled_faces_dict[n2])
  pair=(image1,image2,0)
  negative_pairs.append(pair)

In [19]:
for i in range(10):
  print(f"Negative Pairs: {negative_pairs[i]}")

Negative Pairs: ('/content/lfw_funneled/John_Lynch/John_Lynch_0001.jpg', '/content/lfw_funneled/Luis_Berrondo/Luis_Berrondo_0001.jpg', 0)
Negative Pairs: ('/content/lfw_funneled/Ernie_Harwell/Ernie_Harwell_0001.jpg', '/content/lfw_funneled/Hiroki_Gomi/Hiroki_Gomi_0001.jpg', 0)
Negative Pairs: ('/content/lfw_funneled/Virgina_Ruano_Pascal/Virgina_Ruano_Pascal_0001.jpg', '/content/lfw_funneled/Colleen_Atwood/Colleen_Atwood_0001.jpg', 0)
Negative Pairs: ('/content/lfw_funneled/Jason_Biggs/Jason_Biggs_0001.jpg', '/content/lfw_funneled/Harold_Brown/Harold_Brown_0001.jpg', 0)
Negative Pairs: ('/content/lfw_funneled/Daniel_Rouse/Daniel_Rouse_0001.jpg', '/content/lfw_funneled/Eric_Robert_Rudolph/Eric_Robert_Rudolph_0002.jpg', 0)
Negative Pairs: ('/content/lfw_funneled/Gabrielle_Union/Gabrielle_Union_0001.jpg', '/content/lfw_funneled/Darren_Campel/Darren_Campel_0001.jpg', 0)
Negative Pairs: ('/content/lfw_funneled/Colin_Powell/Colin_Powell_0202.jpg', '/content/lfw_funneled/Gary_Locke/Gary_Locke_

In [20]:
print(len(positive_pairs))
print(len(negative_pairs))

242257
242257


In [21]:
combined_pairs=positive_pairs+negative_pairs
for i in range(20):
  print(f"Combined Pairs: {combined_pairs[-i]}")

Combined Pairs: ('/content/lfw_funneled/Vidar_Helgesen/Vidar_Helgesen_0001.jpg', '/content/lfw_funneled/Vidar_Helgesen/Vidar_Helgesen_0002.jpg', 1)
Combined Pairs: ('/content/lfw_funneled/Mike_Cunning/Mike_Cunning_0001.jpg', '/content/lfw_funneled/Hiroki_Gomi/Hiroki_Gomi_0001.jpg', 0)
Combined Pairs: ('/content/lfw_funneled/Terrence_Kiel/Terrence_Kiel_0001.jpg', '/content/lfw_funneled/Enos_Slaughter/Enos_Slaughter_0001.jpg', 0)
Combined Pairs: ('/content/lfw_funneled/Albaro_Recoba/Albaro_Recoba_0001.jpg', '/content/lfw_funneled/Jeff_Weaver/Jeff_Weaver_0001.jpg', 0)
Combined Pairs: ('/content/lfw_funneled/George_Blaney/George_Blaney_0001.jpg', '/content/lfw_funneled/Robert_Hanssen/Robert_Hanssen_0001.jpg', 0)
Combined Pairs: ('/content/lfw_funneled/Hank_Aaron/Hank_Aaron_0001.jpg', '/content/lfw_funneled/Sasha_Cohen/Sasha_Cohen_0001.jpg', 0)
Combined Pairs: ('/content/lfw_funneled/Michael_Denzel/Michael_Denzel_0001.jpg', '/content/lfw_funneled/Gerald_Fitch/Gerald_Fitch_0001.jpg', 0)
Comb

In [22]:
random.shuffle(combined_pairs)

In [23]:
from PIL import Image

In [24]:
# for index,pair in enumerate(combined_pairs):
#     img1=train_transform(Image.open(pair[0]))
#     img2=train_transform(Image.open(pair[1]))
#     label=pair[2]
#     combined_pairs[index]=(img1, img2, label)

In [35]:
from torch.utils.data import Dataset
from PIL import Image

In [36]:
class SiameseDataset(Dataset):
  def __init__(self,pairs,transform=None) -> None:
     self.pairs=pairs
     self.transform=transform
  def __len__(self):
    return len(self.pairs)
  def __getitem__(self,index):
    img1,img2,label=self.pairs[index]
    img1=Image.open(img1).convert("RGB")
    img2=Image.open(img2).convert("RGB")
    if self.transform:
      img1=self.transform(img1)
      img2=self.transform(img2)
      return img1,img2,torch.tensor(label,dtype=torch.float32)


In [27]:
total=len(combined_pairs)
train_size=int(0.8 * total)
val_size=total-train_size

In [28]:
train_pairs=combined_pairs[:train_size]
val_pairs=combined_pairs[train_size:]

In [29]:
train_dataset=SiameseDataset(train_pairs,transform=train_transform)
val_dataset=SiameseDataset(val_pairs,transform=val_transform)

In [30]:
train_loader=DataLoader(train_dataset,batch_size=32,shuffle=True,num_workers=2,pin_memory=True)
val_loader=DataLoader(val_dataset,batch_size=32,shuffle=False,num_workers=2,pin_memory=True)

In [34]:
img1,img2,label=train_dataset[1]
print(f"Image 1: {img1.shape}")
print(f"Image 2: {img2.shape}")
print(f"Label: {label}")

Image 1: torch.Size([3, 224, 224])
Image 2: torch.Size([3, 224, 224])
Label: 0.0
